# 2.1.5 scatter 和 gather 详解

本 notebook 详细介绍 scatter 和 gather 操作，这是 PyTorch 中最重要但也最容易混淆的操作之一

In [ ]:
import torch
import numpy as np

## 1. gather - 按索引收集元素

`gather` 根据 index 从 input 中收集元素

**语法**: `torch.gather(input, dim, index)`

**规则**: `out[i][j][k] = input[index[i][j][k]][j][k]  # dim=0`

In [ ]:
# gather 基本示例
input_tensor = torch.tensor([[1, 2, 3],
                            [4, 5, 6],
                            [7, 8, 9]])

index = torch.tensor([[0, 1, 0],
                     [2, 0, 1],
                     [1, 2, 0]])

print(f"input:\n{input_tensor}")
print(f"\nindex:\n{index}")

# dim=0: 在行维度上收集
result_dim0 = torch.gather(input_tensor, dim=0, index=index)
print(f"\ngather(dim=0):\n{result_dim0}")
print("\n解释 (dim=0):")
print("result[i,j] = input[index[i,j], j]")
print(f"result[0,0] = input[index[0,0], 0] = input[0, 0] = {input_tensor[0, 0]}")
print(f"result[1,0] = input[index[1,0], 0] = input[2, 0] = {input_tensor[2, 0]}")

In [ ]:
# dim=1: 在列维度上收集
result_dim1 = torch.gather(input_tensor, dim=1, index=index)
print(f"gather(dim=1):\n{result_dim1}")
print("\n解释 (dim=1):")
print("result[i,j] = input[i, index[i,j]]")
print(f"result[0,0] = input[0, index[0,0]] = input[0, 0] = {input_tensor[0, 0]}")
print(f"result[0,1] = input[0, index[0,1]] = input[0, 1] = {input_tensor[0, 1]}")
print(f"result[1,0] = input[1, index[1,0]] = input[1, 2] = {input_tensor[1, 2]}")

## 2. gather 可视化理解

In [ ]:
# 使用简单例子可视化 gather
input_tensor = torch.tensor([[10, 11, 12],
                            [20, 21, 22],
                            [30, 31, 32]])

# 每行选择不同的列
index_cols = torch.tensor([[0],  # 第0行选第0列
                          [1],  # 第1行选第1列
                          [2]]) # 第2行选第2列

result = torch.gather(input_tensor, dim=1, index=index_cols)
print("示例: 提取对角线元素")
print(f"input:\n{input_tensor}")
print(f"\nindex:\n{index_cols}")
print(f"\nresult (对角线):\n{result}")
print(f"展平: {result.view(-1)}")

## 3. scatter - 按索引分散元素

`scatter` 是 `gather` 的逆操作，将 src 中的元素按照 index 分散到 input 中

**语法**: `tensor.scatter_(dim, index, src)`

**规则**: `self[index[i][j][k]][j][k] = src[i][j][k]  # dim=0`

In [ ]:
# scatter 基本示例 - dim=1
input_tensor = torch.zeros(3, 5)
index = torch.tensor([[0, 1, 2, 0, 0],
                     [2, 0, 0, 1, 2],
                     [0, 1, 2, 2, 1]])
src = torch.tensor([[1.0, 2.0, 3.0, 4.0, 5.0],
                    [6.0, 7.0, 8.0, 9.0, 10.0],
                    [11.0, 12.0, 13.0, 14.0, 15.0]])

print("初始 input:")
print(input_tensor)
print(f"\nindex:\n{index}")
print(f"\nsrc:\n{src}")

result = input_tensor.scatter_(dim=1, index=index, src=src)
print(f"\nscatter(dim=1) 后:\n{result}")
print("\n解释 (dim=1):")
print("input[i, index[i,j]] = src[i,j]")
print(f"input[0, index[0,0]] = input[0, 0] = src[0,0] = {src[0,0]}")
print(f"input[0, index[0,1]] = input[0, 1] = src[0,1] = {src[0,1]}")

## 4. scatter dim=0 vs dim=1 对比

In [ ]:
# 相同的输入数据
input_base = torch.zeros(3, 4)
index = torch.tensor([[0, 1, 2, 0],
                      [1, 2, 0, 1],
                      [2, 0, 1, 2]])
src = torch.tensor([[1.0, 2.0, 3.0, 4.0],
                    [5.0, 6.0, 7.0, 8.0],
                    [9.0, 10.0, 11.0, 12.0]])

print("输入数据:")
print(f"index:\n{index}")
print(f"\nsrc:\n{src}")

# dim=0：在行维度上操作
input_dim0 = input_base.clone()
result_dim0 = input_dim0.scatter_(dim=0, index=index, src=src)
print("\n" + "="*60)
print("dim=0 (行维度操作):")
print(result_dim0)
print("\n解释：对于每一列，根据 index 在行维度上填充")
print("例如第0列：index[:,0]=[0,1,2]，src[:,0]=[1,5,9]")
print("→ result[0,0]=1, result[1,0]=5, result[2,0]=9")

# dim=1：在列维度上操作
input_dim1 = input_base.clone()
result_dim1 = input_dim1.scatter_(dim=1, index=index, src=src)
print("\n" + "="*60)
print("dim=1 (列维度操作):")
print(result_dim1)
print("\n解释：对于每一行，根据 index 在列维度上填充")
print("例如第0行：index[0]=[0,1,2,0]，src[0]=[1,2,3,4]")
print("→ result[0,0]=4(最后覆盖), result[0,1]=2, result[0,2]=3")

## 5. scatter 在 3D 张量上的应用

In [ ]:
# 3D scatter 示例
input_tensor = torch.zeros(2, 3, 4)
print("初始 input 形状:", input_tensor.shape)
print("初始 input[0]:\n", input_tensor[0])
print("\n初始 input[1]:\n", input_tensor[1])

# 创建索引张量，指定在 dim=2（列/宽度维度）上的位置
index = torch.tensor([
    # 样本0
    [[0, 1, 2, 3],
     [3, 2, 1, 0],
     [1, 1, 2, 2]],
    # 样本1
    [[2, 0, 1, 3],
     [0, 0, 1, 1],
     [3, 2, 1, 0]]
])

# 创建源数据
src = torch.arange(1, 25).float().reshape(2, 3, 4)
print("\n源数据 src 形状:", src.shape)
print("src[0]:\n", src[0])
print("\nsrc[1]:\n", src[1])

# 执行 scatter 操作
result = input_tensor.scatter_(dim=2, index=index, src=src)
print("\n执行 scatter(dim=2) 后:")
print("result[0]:\n", result[0])
print("\nresult[1]:\n", result[1])

## 6. scatter_add - 累加模式

In [ ]:
# scatter_add: 相同位置的值会相加而不是覆盖
input_tensor = torch.zeros(3, 5)
index = torch.tensor([[0, 1, 2, 0, 0],
                     [2, 0, 0, 1, 2],
                     [0, 1, 2, 2, 1]])
src = torch.ones(3, 5)

print("初始 input:")
print(input_tensor)
print(f"\nindex:\n{index}")
print(f"\nsrc (全1):\n{src}")

result = input_tensor.scatter_add_(dim=1, index=index, src=src)
print(f"\nscatter_add(dim=1) 后:\n{result}")
print("\n解释: 相同位置的值会相加")
print("例如 result[0,0] = 3，因为有3个索引指向了 (0,0) 位置")

## 7. 实际应用：One-hot 编码

In [ ]:
# 使用 scatter 实现 one-hot 编码
num_classes = 5
labels = torch.tensor([2, 0, 4, 1, 3])

# 方法1: 使用 scatter
one_hot = torch.zeros(labels.size(0), num_classes)
one_hot.scatter_(dim=1, index=labels.unsqueeze(1), value=1.0)

print(f"标签: {labels}")
print(f"\nOne-hot 编码:\n{one_hot}")

# 方法2: 使用 scatter 和 src 参数
one_hot2 = torch.zeros(labels.size(0), num_classes)
one_hot2.scatter_(dim=1, index=labels.unsqueeze(1), src=torch.ones(labels.size(0), 1))

print(f"\n使用 src 参数的结果:\n{one_hot2}")
print(f"\n两种方法结果相同: {torch.equal(one_hot, one_hot2)}")

## 8. 实际应用：标签平滑（Label Smoothing）

In [ ]:
# 标签平滑：将 one-hot 编码变得更"软"
num_classes = 5
labels = torch.tensor([2, 0, 4, 1, 3])
smoothing = 0.1

# 初始化为均匀分布
smoothed_labels = torch.full((labels.size(0), num_classes), smoothing / (num_classes - 1))

# 将正确类别的概率设置为 1 - smoothing
smoothed_labels.scatter_(dim=1, index=labels.unsqueeze(1), value=1.0 - smoothing)

print(f"标签: {labels}")
print(f"\n标签平滑后 (smoothing={smoothing}):\n{smoothed_labels}")
print(f"\n每行的和: {smoothed_labels.sum(dim=1)}")
print("注意: 正确类别的概率降低到 0.9，其他类别分配了 0.025")

## 9. gather 的高级应用：提取 top-k 值

In [ ]:
# 使用 gather 提取 top-k 预测值
logits = torch.randn(3, 10)  # 3个样本，10个类别
k = 3

# 获取 top-k 的索引和值
topk_values, topk_indices = torch.topk(logits, k, dim=1)

print(f"logits 形状: {logits.shape}")
print(f"\nlogits:\n{logits}")
print(f"\ntop-{k} 值:\n{topk_values}")
print(f"\ntop-{k} 索引:\n{topk_indices}")

# 验证：使用 gather 重新提取
gathered_values = torch.gather(logits, dim=1, index=topk_indices)
print(f"\n使用 gather 重新提取:\n{gathered_values}")
print(f"\n结果相同: {torch.allclose(topk_values, gathered_values)}")

## 10. scatter 和 gather 的互逆关系

In [ ]:
# 验证 scatter 和 gather 是互逆操作
original = torch.tensor([[1.0, 2.0, 3.0],
                        [4.0, 5.0, 6.0],
                        [7.0, 8.0, 9.0]])

index = torch.tensor([[0, 2, 1],
                     [1, 0, 2],
                     [2, 1, 0]])

print("原始张量:")
print(original)
print(f"\nindex:\n{index}")

# 使用 scatter 分散
scattered = torch.zeros_like(original)
scattered.scatter_(dim=1, index=index, src=original)
print(f"\nscatter 后:\n{scattered}")

# 使用 gather 收集回来
gathered = torch.gather(scattered, dim=1, index=index)
print(f"\ngather 回来:\n{gathered}")

# 验证是否恢复原始数据
print(f"\n恢复成功: {torch.equal(original, gathered)}")

## 11. 总结

### gather
- **作用**: 根据 index 从 input 中收集元素
- **规则**: `out[i][j] = input[i][index[i][j]]` (dim=1)
- **应用**: 提取特定位置的值、top-k 提取

### scatter
- **作用**: 根据 index 将 src 分散到 input 中
- **规则**: `input[i][index[i][j]] = src[i][j]` (dim=1)
- **应用**: one-hot 编码、标签平滑

### 关键要点
1. `dim` 参数决定在哪个维度上进行操作
2. `index` 的形状必须与 `src` 相同（scatter）或与输出相同（gather）
3. `scatter_add` 会累加相同位置的值
4. scatter 和 gather 是互逆操作
5. 理解 dim=0 和 dim=1 的区别很重要

### 记忆技巧
- **gather**: 收集、聚集 → 从大集合中选出小部分
- **scatter**: 分散、散开 → 将元素分散到不同位置
- **dim=0**: 行操作（垂直方向）
- **dim=1**: 列操作（水平方向）

---

## 📝 练习题

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的练习题来巩固知识。

**建议：**
- 先完成基础题，确保理解核心概念
- 进阶题帮助你深入掌握技巧
- 挑战题锻炼综合应用能力

💪 加油！实践是最好的学习方式！